# GSPO Loss

**GSPO: Group Sequence Policy Optimization**

来自 Alibaba Qwen 团队 (arXiv:2507.18071)，被用于 Qwen3 系列模型训练。

## 核心动机

GRPO 使用 **token 级别**重要性采样比率 $w_{i,t} = \frac{\pi_\theta(y_{i,t}|x,y_{i,<t})}{\pi_{\theta_{\text{old}}}(y_{i,t}|x,y_{i,<t})}$，
但这存在根本性问题：

> 重要性采样的核心原理是通过 **多个样本** 的加权来修正分布偏差。
> 而 GRPO 对每个 token 位置单独计算比率（每个 token 仅有一个样本），
> 无法有效修正分布，反而引入高方差噪声。
> 这种噪声随序列长度**累积**，并被裁剪机制**放大**，最终导致模型崩溃（collapse）。

## GSPO 的解决方案

将重要性采样比率从 **token 级别** 提升到 **序列级别**：

$$
s_i(\theta) = \left(\frac{\pi_\theta(y_i|x)}{\pi_{\theta_{\text{old}}}(y_i|x)}\right)^{\frac{1}{|y_i|}}
= \exp\left(\frac{1}{|y_i|} \sum_{t=1}^{|y_i|} \log\frac{\pi_\theta(y_{i,t}|x,y_{i,<t})}{\pi_{\theta_{\text{old}}}(y_{i,t}|x,y_{i,<t})}\right)
$$

这是 token 级别 log-ratio 的**平均值**取 exp，等价于 token 级别比率的**几何平均**。

**关键优势**：
1. 与序列级别奖励对齐（奖励颁发给整条序列）
2. 裁剪在序列级别进行，不同长度序列有统一的数值范围（通过长度归一化）
3. 彻底消除 token 级别重要性权重不均等导致的不稳定性

## GSPO 目标函数

$$
\mathcal{J}_{\text{GSPO}}(\theta) = \mathbb{E}_{x\sim\mathcal{D},\{y_i\}_{i=1}^G\sim\pi_{\theta_{\text{old}}}(\cdot|x)}
\frac{1}{G}\sum_{i=1}^G \left[ \min\left(s_i(\theta)\hat{A}_i,\ \text{clip}\left(s_i(\theta), 1-\varepsilon, 1+\varepsilon\right)\hat{A}_i\right) \right]
$$

其中序列级别重要性比率：
$$
s_i(\theta) = \exp\left(\frac{1}{|y_i|} \sum_{t=1}^{|y_i|} \log\frac{\pi_\theta(y_{i,t}|x,y_{i,<t})}{\pi_{\theta_{\text{old}}}(y_{i,t}|x,y_{i,<t})}\right)
$$

优势函数（与 GRPO 相同）：
$$
\hat{A}_i = \frac{r(x, y_i) - \text{mean}\{r(x, y_j)\}_{j=1}^G}{\text{std}\{r(x, y_j)\}_{j=1}^G}
$$

## GRPO 梯度 vs GSPO 梯度对比

**GRPO 梯度**（略去裁剪）：
$$
\nabla_\theta \mathcal{J}_{\text{GRPO}} = \mathbb{E}\left[\frac{1}{G}\sum_i \hat{A}_i \cdot \frac{1}{|y_i|}\sum_t \frac{\pi_\theta(y_{i,t})}{\pi_{\theta_{\text{old}}}(y_{i,t})} \cdot \nabla_\theta \log \pi_\theta(y_{i,t})\right]
$$
每个 token 有**不同的权重** $w_{i,t}$，权重不均等且积累高方差。

**GSPO 梯度**（略去裁剪）：
$$
\nabla_\theta \mathcal{J}_{\text{GSPO}} = \mathbb{E}\left[\frac{1}{G}\sum_i \hat{A}_i \cdot \underbrace{\left(\frac{\pi_\theta(y_i|x)}{\pi_{\theta_{\text{old}}}(y_i|x)}\right)^{\frac{1}{|y_i|}}}_{s_i(\theta)} \cdot \frac{1}{|y_i|}\sum_t \nabla_\theta \log \pi_\theta(y_{i,t})\right]
$$
同一条响应的所有 token 权重**相同**（都是 $s_i(\theta)$），消除了 token 级别的不均等权重。

## GSPO-token 变体
用于需要 token 级别优势的多轮场景（如 KTO、多轮 RL）：
$$
s_{i,t}(\theta) = \text{sg}[s_i(\theta)] \cdot \frac{\pi_\theta(y_{i,t})}{\text{sg}[\pi_\theta(y_{i,t})]}
$$
其中 $\text{sg}[\cdot]$ 为停止梯度操作，确保梯度只通过当前 token 流动。

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

## 序列级别重要性比率 $s_i(\theta)$ 计算

In [ ]:
def gspo_sequence_ratio(pi_logprob, pi_old_logprob, mask):
    """
    计算 GSPO 序列级别重要性采样比率 s_i(θ)
    
    s_i(θ) = exp(1/|y_i| * sum_t log(π_θ(y_it) / π_θold(y_it)))
           = exp(mean over output tokens of log-ratio)
    
    几何直觉：
    - token-level 比率：r_t = π_θ / π_θold（每个 token 独立）
    - sequence-level 比率：s_i = (∏_t r_t)^(1/|y_i|) = 几何平均值
    
    使用长度归一化（1/|y_i|）的原因：
    - 控制不同长度序列的比率在统一数值范围内
    - 避免长序列的微小 token 变化导致序列比率的剧烈波动
    
    Args:
        pi_logprob: 当前策略 log prob，[bs, seq_len]
        pi_old_logprob: 旧策略 log prob，[bs, seq_len]
        mask: 输出 token 的 mask，[bs, seq_len]，1 表示输出 token
    Returns:
        s: 序列级别 IS 比率，[bs]
    """
    # 每个 token 的 log-ratio: log(π_θ / π_θold)
    log_ratio = pi_logprob - pi_old_logprob  # [bs, seq_len]
    
    # 仅对输出 token 计算（屏蔽 prompt 部分）
    log_ratio_masked = log_ratio * mask  # [bs, seq_len]
    
    # 每条序列的输出 token 数量
    output_lens = mask.sum(dim=1)  # [bs]
    
    # 序列级别比率：log-ratio 的均值再取 exp（几何平均）
    mean_log_ratio = log_ratio_masked.sum(dim=1) / output_lens  # [bs]
    s = torch.exp(mean_log_ratio)  # [bs]，序列级别 IS 比率
    
    return s

## 验证：序列比率 = token 比率的几何平均

In [ ]:
# 手动验证 sequence ratio = geometric mean of token ratios
torch.manual_seed(42)

# 模拟一条序列, 3 个输出 token
pi_logprob_single = torch.tensor([-0.5, -1.2, -0.8, -0.3, -0.9])  # [seq_len=5]
pi_old_logprob_single = torch.tensor([-0.6, -1.0, -0.7, -0.5, -1.1])
mask_single = torch.tensor([0., 0., 1., 1., 1.])  # 前 2 个是输入, 后 3 个是输出

# 方法 1：GSPO 公式计算
pi_logprob_batch = pi_logprob_single.unsqueeze(0)       # [1, 5]
pi_old_logprob_batch = pi_old_logprob_single.unsqueeze(0)
mask_batch = mask_single.unsqueeze(0)
s_gspo = gspo_sequence_ratio(pi_logprob_batch, pi_old_logprob_batch, mask_batch)

# 方法 2：手动计算（几何平均）
output_token_log_ratios = (pi_logprob_single - pi_old_logprob_single)[mask_single == 1]
token_ratios = output_token_log_ratios.exp()  # token 级别比率
geometric_mean = token_ratios.prod() ** (1.0 / len(token_ratios))  # 几何平均

print(f'各输出 token 的 IS 比率：{token_ratios.tolist()}')
print(f'几何平均（手动计算）：{geometric_mean.item():.6f}')
print(f'GSPO s_i（公式计算）：{s_gspo.item():.6f}')
print(f'两者是否相等：{torch.isclose(s_gspo, geometric_mean).item()}')

# 对比：算术平均（GRPO 等效，不是 GSPO）
arithmetic_mean = token_ratios.mean()
print(f'\n对比：token 比率的算术平均（GRPO 等效）：{arithmetic_mean.item():.6f}')
print(f'几何平均 vs 算术平均的差异：{abs(geometric_mean.item() - arithmetic_mean.item()):.6f}')
print('（差异越大说明 token 比率越不均匀，GRPO 的高方差问题越严重）')

## 序列比率 vs Token 比率：方差分析

In [ ]:
# 模拟长序列下两种比率的方差对比
# 这是 GSPO 论文中的核心论点：随序列长度增加，token 比率的方差累积

torch.manual_seed(0)

seq_lengths = [10, 50, 100, 200, 500, 1000]
n_samples = 1000
variance_token = []
variance_seq = []

for seq_len in seq_lengths:
    # 模拟 token 级别的 log-ratio（从标准正态分布采样）
    log_ratios = torch.randn(n_samples, seq_len) * 0.3  # 较小的偏移
    token_ratios = log_ratios.exp()  # [n_samples, seq_len]
    
    # GRPO style: token 级别方差（取各 token 比率的均值）
    mean_token_ratio = token_ratios.mean(dim=1)  # [n_samples]
    variance_token.append(mean_token_ratio.var().item())
    
    # GSPO style: 序列级别方差（几何平均）
    seq_ratio = torch.exp(log_ratios.mean(dim=1))  # [n_samples]
    variance_seq.append(seq_ratio.var().item())

plt.figure(figsize=(10, 4))
plt.plot(seq_lengths, variance_token, 'r-o', label='GRPO: token-level ratio (算术均值)', linewidth=2)
plt.plot(seq_lengths, variance_seq, 'b-o', label='GSPO: sequence-level ratio (几何均值)', linewidth=2)
plt.xlabel('Sequence Length（序列长度）')
plt.ylabel('Variance（方差）')
plt.title('GRPO vs GSPO：IS 比率方差随序列长度的变化')
plt.legend()
plt.grid()
plt.yscale('log')
plt.show()

print('结论：GRPO 的 token 级别比率方差随序列长度增加而增大')
print('      GSPO 的序列级别比率方差保持稳定（受益于中心极限定理）')

## GSPO Loss 核心实现

In [ ]:
def gspo_advantage(rewards):
    """
    组相对优势（与 GRPO 相同）
    
    A_i = (R_i - mean(R)) / std(R)
    """
    epsilon = 1e-5
    return (rewards - rewards.mean()) / (rewards.std() + epsilon)


def gspo_loss(
    pi_logprob,       # 当前策略 log prob，[bs, seq_len]
    pi_old_logprob,   # 旧策略 log prob（stop gradient），[bs, seq_len]
    rewards,          # 组内奖励，[bs]
    input_len,        # prompt 长度
    epsilon=0.2,      # 裁剪阈值（注意：GSPO 的 ε 与 GRPO 的 ε 数量级不同）
    is_debug=True
):
    """
    GSPO Loss 实现
    
    核心思想：将 IS 比率从 token 级别提升到序列级别，
    使用序列似然的几何平均来控制 off-policy 程度。
    
    与 GRPO 的关键区别：
    - GRPO: 对每个 token 单独计算 IS 比率，在 token 级别裁剪
    - GSPO: 整条序列只有一个 IS 比率，在序列级别裁剪
    - 结果：同一序列的所有 token 梯度权重相同（消除 token 级别不均等）
    
    注意：由于 IS 比率定义不同，GSPO 和 GRPO 适合的 epsilon 值也不同。
    GSPO 的序列比率 s_i 通常接近 1，因此可以使用与 GRPO 类似的 epsilon 值。
    """
    bs, seq_len = pi_logprob.shape
    
    # ============================================================
    # Step 1: 构建输出 mask
    # ============================================================
    mask = torch.zeros(bs, seq_len)
    mask[:, input_len:] = 1.0
    
    # ============================================================
    # Step 2: 计算优势（组相对优势，与 GRPO 相同）
    # ============================================================
    advantage = gspo_advantage(rewards)  # [bs]
    
    # ============================================================
    # Step 3: 计算序列级别 IS 比率（GSPO 核心）
    # s_i(θ) = exp(mean_t(log π_θ(y_it) - log π_θold(y_it)))
    # 这是 token 级别 log-ratio 的均值再取 exp
    # ============================================================
    seq_ratio = gspo_sequence_ratio(pi_logprob, pi_old_logprob, mask)  # [bs]
    
    # ============================================================
    # Step 4: 在序列级别进行 PPO-style 裁剪
    # 裁剪发生在整条序列的比率上，而不是每个 token
    # ============================================================
    seq_ratio_clip = torch.clamp(seq_ratio, 1 - epsilon, 1 + epsilon)
    
    # ============================================================
    # Step 5: 计算序列级别的策略梯度目标
    # min(s_i * A_i, clip(s_i) * A_i)
    # 注意：这是序列级别的值，需要广播到所有 token
    # ============================================================
    policy_gradient_seq = torch.minimum(
        seq_ratio * advantage,         # [bs]
        seq_ratio_clip * advantage     # [bs]
    )  # [bs]
    
    # ============================================================
    # Step 6: 计算损失
    # 序列级别的梯度目标均匀分配给该序列的所有 token
    # 每条序列的梯度大小是 policy_gradient_seq[i]，等权重分配给各 token
    # ============================================================
    # 将序列级别值广播到 token 级别（但梯度仍通过均匀权重传播）
    # 等价于：(1/G) * sum_i [min(s_i*Ai, clip(s_i)*Ai)]
    loss = -(1.0 / bs) * policy_gradient_seq.sum()
    
    if is_debug:
        print(f'[Rewards]         : {rewards.tolist()}')
        print(f'[Advantage]       : {advantage.tolist()}')
        print(f'[Seq Ratio s_i]   : {seq_ratio.tolist()}')
        print(f'[Clipped Ratio]   : {seq_ratio_clip.tolist()}')
        print(f'[Loss]            : {loss.item():.6f}')
    
    return loss

## GSPO-token 变体：token 级别优势 + 序列级别权重

In [ ]:
def gspo_token_loss(
    pi_logprob,       # 当前策略 log prob，[bs, seq_len]
    pi_old_logprob,   # 旧策略 log prob，[bs, seq_len]
    rewards,          # 组内奖励，[bs] 或 token 级别奖励 [bs, seq_len]
    input_len,
    epsilon=0.2
):
    """
    GSPO-token 变体
    
    适用于多轮 RL 等需要 token 级别优势的场景。
    
    核心思想：
    - 使用序列级 IS 比率作为固定权重（stop gradient）
    - 梯度通过当前 token 的 log 概率流动
    - s_{i,t}(θ) = sg[s_i(θ)] * π_θ(y_it) / sg[π_θ(y_it)]
    
    等价于：用序列级比率加权的 REINFORCE 目标，但仅允许 token 级梯度流动。
    """
    bs, seq_len = pi_logprob.shape
    
    mask = torch.zeros(bs, seq_len)
    mask[:, input_len:] = 1.0
    
    # 计算序列级别 IS 比率（这部分 stop gradient，不传播梯度）
    with torch.no_grad():
        seq_ratio = gspo_sequence_ratio(pi_logprob, pi_old_logprob, mask)  # [bs]
        seq_ratio_clip = torch.clamp(seq_ratio, 1 - epsilon, 1 + epsilon)  # [bs]
    
    # 优势函数
    advantage = gspo_advantage(rewards).unsqueeze(1)  # [bs, 1]
    
    # token 级别目标：序列级别权重 × log 概率
    # s_{i,t} = sg[clip(s_i)] × (π_θ(y_it) / sg[π_θ(y_it)])
    # 在对数空间：log(s_{i,t}) = sg[log(clip(s_i))] + log(π_θ) - sg[log(π_θ)]
    # 梯度只通过 log(π_θ) 传播（因为另外两项是 stop gradient）
    
    # 简化实现：使用序列级别权重乘以 token 级别 log 概率
    # 梯度：∂/∂θ [seq_ratio_clip * A * 1/|o| * Σ log π_θ]
    #       = seq_ratio_clip * A * 1/|o| * ∂log π_θ/∂θ
    
    output_lens = mask.sum(dim=1)  # [bs]
    
    # 每条序列的序列级权重 × 该序列 token 的 log prob 之和
    token_logprob_sum = (pi_logprob * mask).sum(dim=1)  # [bs]
    policy_gradient = seq_ratio_clip * advantage.squeeze() * token_logprob_sum / output_lens  # [bs]
    
    loss = -(1.0 / bs) * policy_gradient.sum()
    return loss

## 完整测试与验证

In [ ]:
# ======================== 测试 GSPO Loss ========================
torch.manual_seed(0)
bs, seq_len, vocab_size = 4, 8, 32
input_len = 3

pi_logits     = torch.randn(bs, seq_len, vocab_size)
pi_old_logits = torch.randn(bs, seq_len, vocab_size)

pi_logprob_all     = F.log_softmax(pi_logits,     dim=-1)
pi_old_logprob_all = F.log_softmax(pi_old_logits, dim=-1)

token_ids = torch.randint(0, vocab_size, (bs, seq_len))
pi_logprob_all     = torch.gather(pi_logprob_all,     -1, token_ids.unsqueeze(-1)).squeeze(-1)
pi_old_logprob_all = torch.gather(pi_old_logprob_all, -1, token_ids.unsqueeze(-1)).squeeze(-1)

rewards = torch.tensor([1.0, 0.0, 1.0, 0.0])

print('='*50)
print('GSPO Loss 测试')
print('='*50)
loss = gspo_loss(pi_logprob_all, pi_old_logprob_all, rewards, input_len=input_len)

print()
print('='*50)
print('当新旧策略相同时（on-policy），序列比率应为 1')
print('='*50)
loss_onpolicy = gspo_loss(pi_logprob_all, pi_logprob_all.detach(),
                          rewards, input_len=input_len)

## 直觉可视化：序列级别裁剪的效果

In [ ]:
# 可视化 GSPO 裁剪：序列级别比率的分布
# 展示 GSPO 比率相比 GRPO token 比率更稳定

torch.manual_seed(42)
n = 1000
seq_len = 200

# 模拟策略更新后的 log prob 变化
log_ratios = torch.randn(n, seq_len) * 0.5  # 每个 token 的 log-ratio

# GRPO token 比率（每个 token 独立）
token_ratios = log_ratios.exp()  # [n, seq_len]

# GSPO 序列比率（几何平均）
seq_ratios = torch.exp(log_ratios.mean(dim=1))  # [n]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：GRPO token 比率分布
axes[0].hist(token_ratios.flatten().numpy(), bins=100, range=(0, 5), density=True, alpha=0.7, color='red')
axes[0].axvline(x=1, color='k', linestyle='--', label='ratio=1 (on-policy)')
axes[0].axvline(x=1.2, color='orange', linestyle='--', label='upper clip (ε=0.2)')
axes[0].set_title(f'GRPO: Token-level IS Ratio\nstd={token_ratios.std():.3f}')
axes[0].set_xlabel('IS Ratio')
axes[0].legend()
axes[0].grid()

# 右图：GSPO 序列比率分布
axes[1].hist(seq_ratios.numpy(), bins=100, range=(0, 5), density=True, alpha=0.7, color='blue')
axes[1].axvline(x=1, color='k', linestyle='--', label='ratio=1 (on-policy)')
axes[1].axvline(x=1.2, color='orange', linestyle='--', label='upper clip (ε=0.2)')
axes[1].set_title(f'GSPO: Sequence-level IS Ratio\nstd={seq_ratios.std():.3f}')
axes[1].set_xlabel('IS Ratio')
axes[1].legend()
axes[1].grid()

plt.tight_layout()
plt.show()

print(f'GRPO token 比率标准差：{token_ratios.std():.4f}（高方差，不稳定）')
print(f'GSPO 序列比率标准差：{seq_ratios.std():.4f}（低方差，稳定）')
print(f'方差降低比例：{(1 - seq_ratios.std()/token_ratios.std()) * 100:.1f}%')

## GSPO vs GRPO 总结

| 特性 | GRPO | GSPO |
|------|------|------|
| IS 比率单位 | Token 级别 | 序列级别 |
| IS 比率计算 | $w_{i,t} = \pi_\theta / \pi_{\theta_{\text{old}}}$（每个 token 独立） | $s_i = (\prod_t r_t)^{1/|y_i|}$（几何平均） |
| 裁剪粒度 | Token 级别裁剪 | 序列级别裁剪 |
| 同序列 token 权重 | 不等（取决于各 token 的比率） | 相等（都是 $s_i$） |
| 稳定性（长序列）| 方差随长度累积 | 稳定（受益于平均效应） |
| MoE 模型训练 | 容易崩溃 | 稳定 |
| 实际应用 | DeepSeek-R1 等 | Qwen3 系列 |

**核心直觉**：「奖励颁发给整条序列，那么控制 off-policy 程度也应该在序列级别进行」。
GSPO 通过这一简洁的设计原则，从根本上消除了 GRPO 在长序列训练中的不稳定性。